# BarqTrain Colab: Training + Inference

This notebook shows a full Colab workflow:
1. Install BarqTrain
2. Restart the runtime/session after the native build
3. Verify Rust + CUDA runtime loading
4. Fine-tune a small Causal LM for a few steps
5. Run inference

> Recommended runtime: **GPU** (`Runtime -> Change runtime type -> T4/A100`)


In [ ]:
%%bash
set -euo pipefail
if [ ! -x "$HOME/.cargo/bin/cargo" ]; then
  python - <<'INNERPY'
from pathlib import Path
from urllib.request import urlopen
script = urlopen('https://sh.rustup.rs').read().decode('utf-8')
path = Path('/tmp/rustup-init.sh')
path.write_text(script)
INNERPY
  sh /tmp/rustup-init.sh -y >/dev/null
fi
export PATH="$HOME/.cargo/bin:$PATH"
python -m pip install --upgrade pip setuptools wheel setuptools-rust
python -m pip install ninja packaging datasets accelerate peft trl
if [ ! -d /content/BarqTrain ]; then
  git clone https://github.com/YASSERRMD/BarqTrain.git /content/BarqTrain
fi
cd /content/BarqTrain
python -m pip install -e . --no-build-isolation
python - <<'INNERPY'
import importlib.util
assert importlib.util.find_spec('barqtrain_rs'), 'barqtrain_rs did not build; stop here and inspect the cargo output above.'
print('barqtrain_rs build: OK')
INNERPY
if [ -n "${COLAB_GPU:-}" ] || [ -d /usr/local/cuda ]; then
  BARQTRAIN_BUILD_CUDA=1 python -m pip install -e . --no-build-isolation
  python - <<'INNERPY'
import importlib.util
assert importlib.util.find_spec('barqtrain_cuda'), 'barqtrain_cuda did not build; stop here and inspect the CUDA build output above.'
print('barqtrain_cuda build: OK')
INNERPY
fi
cargo --version
echo "Native build finished. Restart the runtime/session now, then continue with the next cell."


In [ ]:
%cd /content/BarqTrain

import importlib
import importlib.util
import os
import sys
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments

os.environ['BARQTRAIN_AUTO_BUILD'] = '0'
sys.path.insert(0, '/content/BarqTrain/python')
importlib.invalidate_caches()
assert importlib.util.find_spec('barqtrain_rs'), 'barqtrain_rs did not build; re-run the install cell and inspect the cargo output.'

from barqtrain import PackedCausalLMDataCollator, patch_model
import barqtrain._ffi as ffi

assert ffi.load_rust_backend() is not None, 'barqtrain_rs failed to load at runtime; stop here and inspect the install output.'
print('Restart completed. Runtime verification follows.')
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('barqtrain_rs available:', bool(importlib.util.find_spec('barqtrain_rs')))
print('barqtrain_cuda available:', bool(importlib.util.find_spec('barqtrain_cuda')))
if torch.cuda.is_available():
    assert importlib.util.find_spec('barqtrain_cuda'), 'barqtrain_cuda did not build; re-run the install cell and inspect the CUDA build output.'
    assert ffi.load_cuda_backend() is not None, 'barqtrain_cuda failed to load at runtime; stop here and inspect the CUDA build output.'


In [ ]:
MODEL_NAME = "Qwen/Qwen2-0.5B-Instruct"
MAX_LEN = 256

# Load tokenizer + model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
dtype = torch.bfloat16 if use_bf16 else (torch.float16 if torch.cuda.is_available() else torch.float32)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=dtype,
    trust_remote_code=True,
)
if torch.cuda.is_available():
    model = model.cuda()

# Apply BarqTrain patching
patch_model(model)
model.config.use_cache = False

print('Model loaded and patched for training.')
                


In [ ]:
raw_ds = load_dataset('wikitext', 'wikitext-2-raw-v1', split='train[:2%]')
raw_ds = raw_ds.filter(lambda example: bool(example['text'] and example['text'].strip()))
raw_ds = raw_ds.shuffle(seed=42)


def tokenize(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        max_length=MAX_LEN,
        padding=False,
    )


tokenized = raw_ds.map(tokenize, batched=True, remove_columns=raw_ds.column_names)
collator = PackedCausalLMDataCollator(
    max_length=MAX_LEN,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
args = TrainingArguments(
    output_dir='/content/barqtrain_colab_demo',
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=2e-5,
    max_steps=20,
    logging_steps=5,
    fp16=torch.cuda.is_available() and not use_bf16,
    bf16=use_bf16,
    report_to=[],
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized,
    data_collator=collator,
)

trainer.train()


In [ ]:
prompt = "Write 3 tips to optimize LLM fine-tuning speed:" 
inputs = tokenizer(prompt, return_tensors="pt")
if torch.cuda.is_available():
    inputs = {k: v.cuda() for k, v in inputs.items()}

model.eval()
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=80, do_sample=True, temperature=0.7)

print(tokenizer.decode(out[0], skip_special_tokens=True))
                
